<a href="https://colab.research.google.com/github/jennifersolano308-cyber/AAI2025/blob/main/Part_1_Prompt_Chaining_for_a_Customer_Support_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install openai

In [12]:
import os, json
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

client = OpenAI()
MODEL = "gpt-4o-mini"

def call_llm(system, user, json_mode=False, temperature=0.2):
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        **kwargs)
    return r.choices[0].message.content

In [13]:
def run_chain(message, customer_followup, verbose=True):
    log = {}
    # Step 1: classify
    triage = json.loads(call_llm(STEP1_SYSTEM, message, json_mode=True))
    log["step1"] = triage

    # Step 2: clarifying questions (depends on triage["missing_info"])
    s2 = call_llm(STEP2_SYSTEM, f"Message: {message}\nTriage: {json.dumps(triage)}")
    log["step2"] = s2

    # Step 3: solution (depends on triage + customer answers)
    s3_in = (f"Message: {message}\nTriage: {json.dumps(triage)}\n"
             f"Questions asked: {s2}\nCustomer answers: {customer_followup}")
    solution = json.loads(call_llm(STEP3_SYSTEM, s3_in, json_mode=True))
    log["step3"] = solution

    # Step 4: escalation rule (depends on triage + solution)
    s4_in = f"Triage: {json.dumps(triage)}\nSolution: {json.dumps(solution)}\nOriginal message: {message}"
    esc = json.loads(call_llm(STEP4_SYSTEM, s4_in, json_mode=True))
    log["step4"] = esc

    # Step 5: final reply (depends on all prior outputs)
    s5_in = f"Triage: {json.dumps(triage)}\nSolution: {json.dumps(solution)}\nEscalation: {json.dumps(esc)}"
    log["step5"] = call_llm(STEP5_SYSTEM, s5_in, temperature=0.4)

    if verbose:
        print("CUSTOMER:", message, "\n")
        for k in ["step1", "step2", "step3", "step4", "step5"]:
            v = log[k]
            print(f"--- {k.upper()} ---")
            print(json.dumps(v, indent=2) if isinstance(v, dict) else v, "\n")
    return log